# 📈 Project 08: Financial News Sentiment & Market Volatility Forecasting
### Quantitative Finance, Domain NLP (FinBERT) & Hybrid Volatility Forecasting

**Author:** Data Science Portfolio Team  
**Difficulty:** 🟡 Intermediate  
**Domain:** Quantitative Finance  

---
### Notebook Outline:
1. **Environment Setup**
2. **Ingestion of Daily Financial & News Sentiment Data**
3. **EDA: Sentiment Polarity vs. Market Volatility Distribution**
4. **Feature Engineering: Rolling Volatility & Lagged Sentiment Shocks**
5. **Model Benchmarking: Historical Volatility vs. Sentiment-Enriched Ensemble**
6. **Feature Importance: Quantifying the Marginal Value of News**

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import lightgbm as lgb
from sklearn.metrics import mean_squared_error, mean_absolute_error

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
print("Quantitative finance environment ready.")

In [ ]:
# Ingestion
df = pd.read_csv("data/financial_news_volatility.csv")
df['date'] = pd.to_datetime(df['date'])
print(f"Trading Days: {len(df)}")
display(df.head(4))

In [ ]:
# Relationship between News Sentiment and Volatility
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.scatterplot(data=df, x='finbert_sentiment_polarity', y='realized_volatility', 
                hue='vix_close', palette='plasma', ax=axes[0])
axes[0].set_title("FinBERT Sentiment vs. Realized Volatility", fontweight='bold')

sns.lineplot(data=df.tail(60), x='date', y='realized_volatility', ax=axes[1], label='Realized Vol', color='black')
ax_twin = axes[1].twinx()
sns.lineplot(data=df.tail(60), x='date', y='finbert_sentiment_polarity', ax=ax_twin, label='Sentiment', color='red', alpha=0.6)
axes[1].set_title("Volatility Spikes vs. Sentiment Dips (Last 60 Days)", fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Feature Engineering & Time-Series Split
df['vol_lag1'] = df['realized_volatility'].shift(1)
df['vol_lag5'] = df['realized_volatility'].shift(5)
df['sentiment_lag1'] = df['finbert_sentiment_polarity'].shift(1)
df['vix_lag1'] = df['vix_close'].shift(1)

df_model = df.dropna().reset_index(drop=True)

split_point = int(len(df_model) * 0.8)
train = df_model.iloc[:split_point]
test = df_model.iloc[split_point:]

# Baseline Model: Volatility Lag-1
base_rmse = np.sqrt(mean_squared_error(test['realized_volatility'], test['vol_lag1']))

# Sentiment-Enriched LightGBM
features = ['vol_lag1', 'vol_lag5', 'sentiment_lag1', 'vix_lag1', 'news_headline_volume']
model = lgb.LGBMRegressor(n_estimators=100, max_depth=4, learning_rate=0.03, random_state=42, verbose=-1)
model.fit(train[features], train['realized_volatility'])

preds = model.predict(test[features])
lgb_rmse = np.sqrt(mean_squared_error(test['realized_volatility'], preds))

print("=== Volatility Forecasting Benchmarks ===")
print(f"Historical Lag Baseline RMSE: {base_rmse:.4f}")
print(f"Sentiment-Augmented LGBM RMSE: {lgb_rmse:.4f} (Reduction: {((base_rmse - lgb_rmse)/base_rmse):.2%})")